# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the FAIR<sup>2</sup> dataset on protein abundance and modifications from human extracellular vesicles, using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"Dataset Loaded: {getattr(metadata, 'name', '')}\n\nDescription: {getattr(metadata, 'description', '')}")
# Optionally, print identifier, publication date, and license
print(f"Identifier: {getattr(metadata, 'identifier', '')}")
print(f"License: {getattr(metadata, 'license', '')}")
print(f"Date Published: {getattr(metadata, 'datePublished', '')}")

## 2. Data Overview
Review available record sets, fields, and their `@id` identifiers.

In [ ]:
# List all record sets in the dataset
print("Available Record Sets (by @id and name):")
for rs in dataset.metadata.recordSets:
    print(f"  - @id: {rs['@id']}, name: {rs.get('name', '')}")

# As an example, list the first record set's fields and columns
record_sets = [rs["@id"] for rs in dataset.metadata.recordSets]
if record_sets:
    selected_rs_id = record_sets[0]
    rs_obj = next(rs for rs in dataset.metadata.recordSets if rs['@id'] == selected_rs_id)
    print(f"\nFields for RecordSet {selected_rs_id}:")
    for field in rs_obj.get('field', []):
        print(f"  - @id: {field['@id']}, name: {field.get('name','')} dataType: {field.get('dataType','')}")

    print(f"\nColumns for RecordSet {selected_rs_id}:")
    for col in rs_obj.get('column', []):
        print(f"  - @id: {col['@id']}, name: {col.get('name','')}")

## 3. Data Extraction
Load data from each main record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from main record sets
record_sets = [rs['@id'] for rs in dataset.metadata.recordSets]  # collect all recordSet @ids
dataframes = {}

for record_set in record_sets:
    print(f"Loading records for record set: {record_set}")
    try:
        records = list(dataset.records(record_set=record_set))
        df = pd.DataFrame(records)
        dataframes[record_set] = df
        print(f"Loaded {len(df)} rows and {len(df.columns)} columns.")
    except Exception as e:
        print(f"Could not load record set {record_set}: {e}")

# Show columns of the first record set as an example
if record_sets:
    example_rs_id = record_sets[0]
    print(f"\nColumns in '{example_rs_id}':")
    print(dataframes[example_rs_id].columns.tolist())
    dataframes[example_rs_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes removing outliers, transforming data distributions, or grouping data by key attributes to prepare for further analysis.

In [ ]:
# Select a numeric field for demonstration
# Adapt the field '@id' to your use case as discovered above

example_rs_id = record_sets[0] if record_sets else None
df = dataframes[example_rs_id]

# Identify a numeric column for demonstration (e.g., 'coverage', 'MW', or 'PeptideCount')
candidate_numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
print("Candidate numeric fields:", candidate_numeric_fields)

# For demonstration, choose the first available numeric field
if candidate_numeric_fields:
    numeric_field = candidate_numeric_fields[0]
    threshold = df[numeric_field].median()

    # Filter
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold}:")
    print(filtered_df.head())

    # Normalize
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"\nNormalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try to find a grouping field (e.g., 'description', 'Sample', 'protein', etc.)
    candidate_group_fields = [col for col in df.columns if (df[col].dtype == 'object' or pd.api.types.is_categorical_dtype(df[col])) and col != numeric_field]
    group_field = candidate_group_fields[0] if candidate_group_fields else None
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"\nGrouped data by {group_field}:")
        print(grouped_df.head())
else:
    print("No numeric fields found for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
# Basic plotting using pandas and matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

if example_rs_id and candidate_numeric_fields:
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    # If we have a group field, plot mean per group
    if group_field:
        plt.figure(figsize=(10,4))
        order = grouped_df.sort_values(numeric_field, ascending=False)[group_field]
        sns.barplot(data=grouped_df, x=group_field, y=numeric_field, order=order[:10])
        plt.title(f"Mean {numeric_field} per {group_field} (top 10)")
        plt.xlabel(group_field)
        plt.ylabel(f"Mean {numeric_field}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
We have successfully loaded and explored the FAIR<sup>2</sup> dataset using the Croissant schema and the `mlcroissant` library.

- The dataset provides comprehensive information about protein abundance and modifications derived from human extracellular vesicles, with multiple record sets available for analysis.
- We reviewed metadata, loaded record sets dynamically by their `@id`, and demonstrated basic filtering, normalization, and visualization techniques.
- Further analysis can extend to more advanced machine learning and statistical pipelines, leveraging the well-annotated Croissant data structure.

*Tip: For advanced exploration, use the specific field and record set `@id`s from the overview step for robust and reproducible analyses.*